# Alytes-ReID — Setup & Training

**Who runs this notebook**: researcher / developer (once per project, or when new labeled data arrives).

**What it does**:
1. Install dependencies
2. Download toad photos from iNaturalist (no account needed)
3. Auto-annotate images using YOLO-World (open-vocabulary detector)
4. Fine-tune YOLO11 on toad-specific data
5. Validate SAM2 segmentation
6. Train Re-ID model (when labeled data is available)
7. Save models to Google Drive → used by `02_toad_reid.ipynb`

**No manual annotation required** — YOLO-World finds toads by text description ("toad", "frog").

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danort92/Alytes-ReID/blob/claude/alytes-reid-system-qgved/notebooks/01_setup_and_training.ipynb)

---
## 1. Environment Setup

Run this cell once. It installs everything needed and connects Google Drive.

In [ ]:
# Check GPU
!nvidia-smi

# Clone repo
!git clone -b claude/alytes-reid-system-qgved https://github.com/danort92/Alytes-ReID.git 2>/dev/null || (cd Alytes-ReID && git pull)
%cd Alytes-ReID

# Install dependencies
!pip install -r requirements.txt -q

# Mount Google Drive (to save trained models)
from google.colab import drive
drive.mount('/content/drive')

DRIVE_MODEL_DIR = '/content/drive/MyDrive/Alytes-ReID/models'
!mkdir -p {DRIVE_MODEL_DIR}

print('\nSetup complete!')

---
## 2. Download Toad Photos from iNaturalist

Downloads research-grade photos of *Alytes obstetricans* from iNaturalist.  
No account or API key needed — the iNaturalist API is public.

In [ ]:
from pathlib import Path
from src.data.download_inat import download_alytes_images

MAX_IMAGES = 500  # increase to 1000+ for better results (takes longer)

inat_images = download_alytes_images(
    output_dir=Path('data/raw/inaturalist'),
    max_images=MAX_IMAGES,
)
print(f'\nDownloaded {len(inat_images)} images')

---
## 3. Preview Downloaded Images

In [ ]:
import random
import cv2
import matplotlib.pyplot as plt

sample = random.sample(inat_images, min(8, len(inat_images)))

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, img_path in zip(axes.flat, sample):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(img_path.name[:20], fontsize=8)
    ax.axis('off')
plt.suptitle('Sample iNaturalist Images (Alytes obstetricans)')
plt.tight_layout()
plt.show()

print(f'Total images available: {len(inat_images)}')

---
## 4. Auto-Annotate Images

Uses **YOLO-World** (open-vocabulary detector) to find toads/frogs in photos
by text description — much more robust than fixed COCO classes, which miss
camouflaged toads in field photos.

No manual annotation needed! Detections are saved as YOLO-format label files.

In [ ]:
# Ensure ultralytics is installed (may not be present in Colab by default)
!pip install ultralytics -q

from src.data.auto_annotate import auto_annotate

stats = auto_annotate(
    images_dir=Path('data/raw/inaturalist'),
    output_dir=Path('data/raw/auto_labels'),
    model_name='yolov8s-worldv2',  # open-vocabulary detector
    confidence=0.10,               # low threshold — wildlife photos are hard
    text_classes=['toad', 'frog'],  # text prompts for detection
)

print(f'\n=== Auto-Annotation Results ===')
print(f"  Images processed:  {stats['total_images']}")
print(f"  Images annotated:  {stats['annotated']}")
print(f"  Images skipped:    {stats['skipped']}  (no toad/frog detected)")
print(f"  Total bounding boxes: {stats['total_boxes']}")

### Preview auto-annotations

Visual check: do the bounding boxes look correct?

In [ ]:
from src.utils.visualization import draw_yolo_labels

# Find images that got annotated
label_dir = Path('data/raw/auto_labels')
annotated_images = [
    img for img in inat_images
    if (label_dir / f'{img.stem}.txt').exists()
]

preview = random.sample(annotated_images, min(6, len(annotated_images)))

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, img_path in zip(axes.flat, preview):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    label_path = label_dir / f'{img_path.stem}.txt'
    img_with_boxes = draw_yolo_labels(img, label_path)
    ax.imshow(img_with_boxes)
    ax.set_title(img_path.name[:25], fontsize=8)
    ax.axis('off')
plt.suptitle('Auto-annotated images (green boxes = detected toads)')
plt.tight_layout()
plt.show()

---
## 5. Prepare Dataset for Training

Splits auto-annotated images into train/val/test sets in YOLO format.

In [ ]:
from src.data.prepare_dataset import split_dataset, create_yolo_dataset_yaml

counts = split_dataset(
    images_dir=Path('data/raw/inaturalist'),
    labels_dir=Path('data/raw/auto_labels'),
    output_dir=Path('data/processed/detection'),
)

dataset_yaml = create_yolo_dataset_yaml(
    dataset_dir=Path('data/processed/detection'),
    classes=['toad'],
)

print(f'\nDataset splits: {counts}')
print(f'Dataset config: {dataset_yaml}')

---
## 6. Fine-Tune YOLO11 for Toad Detection

Fine-tunes YOLO11 on the auto-annotated toad dataset.  
Starting from COCO weights gives the model a strong head start.

In [ ]:
# Training settings — adjust if needed
YOLO_MODEL = 'yolo11s'  # yolo11n (fastest) | yolo11s | yolo11m | yolo11l | yolo11x
EPOCHS = 100
BATCH_SIZE = 16  # reduce to 8 if you get GPU out-of-memory errors

print(f'Model: {YOLO_MODEL}, Epochs: {EPOCHS}, Batch size: {BATCH_SIZE}')

In [ ]:
from src.detection.train import load_config, train_detector

config = load_config(Path('config/detection.yaml'))

# Override config with the values set above
config['model']['architecture'] = YOLO_MODEL
config['training']['epochs'] = EPOCHS
config['training']['batch_size'] = BATCH_SIZE

# Train
best_weights = train_detector(config)
print(f'\nBest weights saved to: {best_weights}')

---
## 7. Evaluate Detection

In [ ]:
from src.detection.evaluate import evaluate_model

metrics = evaluate_model(best_weights, config)

print('\n=== Detection Results ===')
for name, value in metrics.items():
    print(f'  {name:15s}: {value:.4f}')

---
## 8. SAM2 Segmentation Validation

In [ ]:
# Install SAM2 (not in requirements.txt to keep Colab install light)
!pip install segment-anything-2 -q

In [ ]:
from src.segmentation.segment import ToadSegmenter
from src.detection.predict import load_detector, detect_toads, get_best_detection
from src.utils.visualization import draw_detections, draw_mask_overlay

detector = load_detector(best_weights)
segmenter = ToadSegmenter()  # loads SAM2 lazily on first call

# Pick a test image
test_path = inat_images[0]
test_img = cv2.cvtColor(cv2.imread(str(test_path)), cv2.COLOR_BGR2RGB)

detections = detect_toads(detector, test_path)
best_det = get_best_detection(detections)

if best_det:
    cropped, mask = segmenter.segment_and_crop(test_img, best_det['bbox'])
    overlay = draw_mask_overlay(
        draw_detections(test_img, detections),
        segmenter.segment_from_bbox(test_img, best_det['bbox'])
    )
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(test_img); axes[0].set_title('Original')
    axes[1].imshow(overlay); axes[1].set_title('Detection + Mask')
    axes[2].imshow(cropped); axes[2].set_title('Cropped')
    for ax in axes: ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No toad detected in test image.')

---
## 9. Re-ID Model Training

Requires labeled data organized as `data/processed/reid/<INDIVIDUAL_ID>/<image>.png`.  
Upload the biologist's photos and run the preparation script, then train.

In [ ]:
# Upload labeled re-ID data from local machine
# from google.colab import files
# uploaded = files.upload()  # upload a zip archive
# !unzip -q your_labeled_data.zip -d data/processed/reid/

# Or copy from Drive:
# !cp -r '/content/drive/MyDrive/Alytes-ReID/reid_data' data/processed/reid/

In [ ]:
# Train Re-ID model
# from src.reid.train import train_reid
#
# reid_config = load_config(Path('config/reid.yaml'))
# reid_model_path = train_reid(reid_config, data_dir=Path('data/processed/reid'))
# print(f'Re-ID model saved to: {reid_model_path}')

---
## 10. Save Models to Google Drive

In [ ]:
import shutil

# Detection model
shutil.copy2(str(best_weights), f'{DRIVE_MODEL_DIR}/detection_best.pt')
print(f'Detection model saved to Drive: {DRIVE_MODEL_DIR}/detection_best.pt')

# Re-ID model (uncomment after training)
# shutil.copy2(str(reid_model_path), f'{DRIVE_MODEL_DIR}/reid_model.pt')

# Re-ID database (uncomment after building)
# shutil.copytree('data/models/reid', f'{DRIVE_MODEL_DIR}/reid_db', dirs_exist_ok=True)

print('Models saved to Google Drive. Ready to use in 02_toad_reid.ipynb')